In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [9]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [10]:
v1.dot(dv)

np.float32(0.32332397)

In [11]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [12]:
v2.dot(dv)

np.float32(0.019730574)

In [13]:
from ingest import load_faq_data
documents = load_faq_data()

In [14]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [15]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [16]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

scores

[np.float32(0.48740587),
 np.float32(0.2099193),
 np.float32(0.76294106),
 np.float32(0.44378525),
 np.float32(0.26083997),
 np.float32(0.4866516),
 np.float32(0.30061552),
 np.float32(0.56009984),
 np.float32(0.45960498),
 np.float32(0.25628167),
 np.float32(0.33153272),
 np.float32(0.27318522),
 np.float32(0.27691635),
 np.float32(0.34122986),
 np.float32(0.46007156),
 np.float32(0.26240277),
 np.float32(0.39200056),
 np.float32(0.10854157),
 np.float32(0.27567303),
 np.float32(0.16646828),
 np.float32(0.31437922),
 np.float32(0.0066854246),
 np.float32(0.11205028),
 np.float32(0.21905681),
 np.float32(0.3400085),
 np.float32(0.23571287),
 np.float32(0.32714826),
 np.float32(0.15088344),
 np.float32(0.16563272),
 np.float32(0.05545033),
 np.float32(0.23541182),
 np.float32(0.08533001),
 np.float32(-0.0030899644),
 np.float32(-0.042598374),
 np.float32(-0.060277097),
 np.float32(0.006491136),
 np.float32(0.034277484),
 np.float32(-0.049589746),
 np.float32(-0.0006707795),
 np.float32(

In [17]:
import numpy as np
X = np.array(vectors)
scores = X.dot(v1)
scores

array([ 0.48740587,  0.2099193 ,  0.762941  , ..., -0.08637964,
        0.03759784, -0.03037035], shape=(1350,), dtype=float32)

In [18]:
idx = np.argmax(scores)
idx, scores[idx]
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [19]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [20]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [24]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [22]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [25]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [26]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [27]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

In [28]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [29]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [30]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join after the course has begun.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'